# Calgary Traffic Risk Intelligence

## 04 — Weather and Risk Analysis

This notebook adds historical hourly weather to the Calgary traffic-incident
analysis and builds the project's transparent **Historical Incident Risk Index**.

### Objectives

- Download and cache official Environment and Climate Change Canada hourly weather
- Clean Calgary International Airport weather observations
- Align weather observations with incident timestamps
- Compare incident frequency across weather conditions using hours of exposure
- Create weather-related incident features
- Build time-specific hotspot profiles
- Create a transparent traffic-control-level Historical Incident Risk Index
- Export reusable datasets for modelling and the Streamlit application

### Weather Source

Environment and Climate Change Canada (ECCC), Historical Climate Data.

Calgary International Airport station:

- Climate ID: `3031092`
- ECCC Station ID used for bulk hourly download: `50430`
- Station: `CALGARY INTL A`
- Transport Canada ID: `YYC`

Historical Climate Data:
https://climate.weather.gc.ca/

Hourly data are downloaded one month at a time and cached locally so the notebook
does not repeatedly download the same files.

> The airport station is used as a citywide weather proxy. Conditions can differ
> across Calgary, so weather relationships should be interpreted as associations,
> not precise weather exposure for every incident location.

In [1]:
from pathlib import Path
from io import StringIO
import time
import re

import numpy as np
import pandas as pd
import plotly.express as px
import requests
from IPython.display import display

# Show useful columns clearly.
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 170)

## Project Paths

This notebook expects the spatial outputs from
`03_hotspot_and_intersection_analysis.ipynb`.

In [2]:
# Find the project root.
cwd = Path.cwd()

if cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEAN_DIR = PROJECT_ROOT / "data" / "cleaned"
WEATHER_RAW_DIR = RAW_DIR / "weather"

INCIDENTS_SPATIAL_FILE = (
    CLEAN_DIR / "incidents_hotspots_intersections.csv"
)

HOTSPOT_SUMMARY_FILE = (
    CLEAN_DIR / "hotspot_summary.csv"
)

CONTROL_SUMMARY_FILE = (
    CLEAN_DIR / "traffic_control_incident_summary.csv"
)

WEATHER_RAW_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Spatial incidents file exists:", INCIDENTS_SPATIAL_FILE.exists())
print("Hotspot summary exists:", HOTSPOT_SUMMARY_FILE.exists())
print("Traffic-control summary exists:", CONTROL_SUMMARY_FILE.exists())

Project root: c:\Users\ahnaf\Desktop\calgary-traffic-risk-intelligence
Spatial incidents file exists: True
Hotspot summary exists: True
Traffic-control summary exists: True


In [3]:
# Stop early if notebook 03 has not been completed.
required_files = [
    INCIDENTS_SPATIAL_FILE,
    HOTSPOT_SUMMARY_FILE,
    CONTROL_SUMMARY_FILE,
]

missing_files = [
    path for path in required_files
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Run 03_hotspot_and_intersection_analysis.ipynb first. "
        f"Missing: {missing_files}"
    )

## Load Spatial Incident Data

In [4]:
# Load outputs from the previous project stage.
incidents = pd.read_csv(INCIDENTS_SPATIAL_FILE)
hotspot_summary = pd.read_csv(HOTSPOT_SUMMARY_FILE)
traffic_controls = pd.read_csv(CONTROL_SUMMARY_FILE)

# Restore incident datetimes.
incidents["start_dt_utc"] = pd.to_datetime(
    incidents["start_dt_utc"],
    utc=True,
)

incidents["start_dt_local"] = pd.to_datetime(
    incidents["start_dt_local"],
    utc=True,
).dt.tz_convert("America/Edmonton")

incidents["modified_dt_utc"] = pd.to_datetime(
    incidents["modified_dt_utc"],
    utc=True,
    errors="coerce",
)

print(f"Incidents: {len(incidents):,}")
print(f"Hotspots: {len(hotspot_summary):,}")
print(f"Traffic controls: {len(traffic_controls):,}")

Incidents: 64,798
Hotspots: 696
Traffic controls: 1,791


## Download Historical Hourly Weather

ECCC's Historical Climate Data service provides hourly data one month at a time.
The code below downloads only missing months and saves each response inside
`data/raw/weather/`.

If the files are already present, the download step simply reuses them.

In [5]:
# ECCC hourly weather download settings.
ECCC_STATION_ID = 50430
ECCC_BASE_URL = (
    "https://climate.weather.gc.ca/"
    "climate_data/bulk_data_e.html"
)

# Use the incident period to determine required weather months.
# Drop timezone information only for the calendar-month calculation.
weather_start = (
    incidents["start_dt_local"]
    .min()
    .tz_localize(None)
    .to_period("M")
)

weather_end = (
    incidents["start_dt_local"]
    .max()
    .tz_localize(None)
    .to_period("M")
)

weather_months = pd.period_range(
    weather_start,
    weather_end,
    freq="M",
)

print("Weather start:", weather_start)
print("Weather end:", weather_end)
print("Months required:", len(weather_months))

Weather start: 2016-12
Weather end: 2026-09
Months required: 118


In [6]:
def download_weather_month(period, overwrite=False):
    """Download one month of ECCC hourly weather and cache it locally."""

    output_file = (
        WEATHER_RAW_DIR
        / f"calgary_weather_{period.year}_{period.month:02d}.csv"
    )

    # Reuse a previously downloaded month.
    if output_file.exists() and not overwrite:
        return output_file

    params = {
        "format": "csv",
        "stationID": ECCC_STATION_ID,
        "Year": period.year,
        "Month": period.month,
        "Day": 1,
        "timeframe": 1,
        "submit": "Download Data",
    }

    response = requests.get(
        ECCC_BASE_URL,
        params=params,
        timeout=60,
    )
    response.raise_for_status()

    text = response.content.decode(
        "utf-8-sig",
        errors="replace",
    )

    # A valid response should look like CSV data.
    if "," not in text or len(text) < 100:
        raise ValueError(
            f"Unexpected ECCC response for {period}."
        )

    output_file.write_text(
        text,
        encoding="utf-8",
    )

    return output_file

In [7]:
# Download only weather months that are not already cached.
downloaded_files = []

for index, period in enumerate(weather_months, start=1):
    path = download_weather_month(period)
    downloaded_files.append(path)

    if index == 1 or index % 12 == 0 or index == len(weather_months):
        print(
            f"Prepared {index}/{len(weather_months)} months "
            f"through {period}"
        )

    # Small delay when a new request is needed.
    time.sleep(0.05)

print("Weather files ready:", len(downloaded_files))

Prepared 1/118 months through 2016-12
Prepared 12/118 months through 2017-11
Prepared 24/118 months through 2018-11
Prepared 36/118 months through 2019-11
Prepared 48/118 months through 2020-11
Prepared 60/118 months through 2021-11
Prepared 72/118 months through 2022-11
Prepared 84/118 months through 2023-11
Prepared 96/118 months through 2024-11
Prepared 108/118 months through 2025-11
Prepared 118/118 months through 2026-09
Weather files ready: 118


## Load and Standardize ECCC Weather Data

ECCC column names can change slightly over time. The helper below searches for
expected fields by several known names instead of assuming one exact schema.

In [8]:
# Load every cached monthly weather CSV.
weather_parts = []

for file in downloaded_files:
    month_df = pd.read_csv(
        file,
        low_memory=False,
    )

    if not month_df.empty:
        weather_parts.append(month_df)

if not weather_parts:
    raise ValueError("No weather data were loaded.")

weather_raw = pd.concat(
    weather_parts,
    ignore_index=True,
)

print(
    f"Raw weather: {weather_raw.shape[0]:,} rows × "
    f"{weather_raw.shape[1]} columns"
)

display(weather_raw.head())

Raw weather: 86,184 rows × 31 columns


,Longitude (x),Latitude (y),Station Name,Climate ID,Date/Time (LST),Year,Month,Day,Time (LST),Flag,Temp (°C),Temp Flag,Dew Point Temp (°C),Dew Point Temp Flag,Rel Hum (%),Rel Hum Flag,Precip. Amount (mm),Precip. Amount Flag,Wind Dir (10s deg),Wind Dir Flag,Wind Spd (km/h),Wind Spd Flag,Visibility (km),Visibility Flag,Stn Press (kPa),Stn Press Flag,Hmdx,Hmdx Flag,Wind Chill,Wind Chill Flag,Weather
0,-114.01,51.12,CALGARY INTL A,3031092,2016-12-01 00:00,2016,12,1,00:00,NaN,-4.8,NaN,-6.1,NaN,91.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,6.4,NaN,88.32,NaN,NaN,NaN,NaN,NaN,Snow
1,-114.01,51.12,CALGARY INTL A,3031092,2016-12-01 01:00,2016,12,1,01:00,NaN,-4.8,NaN,-5.9,NaN,92.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,6.4,NaN,88.34,NaN,NaN,NaN,NaN,NaN,Snow
2,-114.01,51.12,CALGARY INTL A,3031092,2016-12-01 02:00,2016,12,1,02:00,NaN,-4.4,NaN,-5.5,NaN,92.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,3.2,NaN,88.39,NaN,NaN,NaN,NaN,NaN,Snow
3,-114.01,51.12,CALGARY INTL A,3031092,2016-12-01 03:00,2016,12,1,03:00,NaN,-3.7,NaN,-4.7,NaN,93.0,NaN,NaN,NaN,31.0,NaN,5.0,NaN,4.0,NaN,88.40,NaN,NaN,NaN,-6.0,NaN,Snow
4,-114.01,51.12,CALGARY INTL A,3031092,2016-12-01 04:00,2016,12,1,04:00,NaN,-3.5,NaN,-4.8,NaN,91.0,NaN,NaN,NaN,22.0,NaN,5.0,NaN,4.8,NaN,88.43,NaN,NaN,NaN,-6.0,NaN,Snow


In [9]:
# Review source column names once.
for column in weather_raw.columns:
    print(column)

Longitude (x)
Latitude (y)
Station Name
Climate ID
Date/Time (LST)
Year
Month
Day
Time (LST)
Flag
Temp (°C)
Temp Flag
Dew Point Temp (°C)
Dew Point Temp Flag
Rel Hum (%)
Rel Hum Flag
Precip. Amount (mm)
Precip. Amount Flag
Wind Dir (10s deg)
Wind Dir Flag
Wind Spd (km/h)
Wind Spd Flag
Visibility (km)
Visibility Flag
Stn Press (kPa)
Stn Press Flag
Hmdx
Hmdx Flag
Wind Chill
Wind Chill Flag
Weather


In [10]:
def find_column(columns, candidates):
    """Return the first matching column name from a candidate list."""

    normalized = {
        str(column).strip().lower(): column
        for column in columns
    }

    for candidate in candidates:
        key = candidate.strip().lower()

        if key in normalized:
            return normalized[key]

    return None


# Locate fields used by the project.
weather_columns = {
    "datetime_utc": find_column(
        weather_raw.columns,
        [
            "Date/Time (UTC)",
            "Date/Time UTC",
        ],
    ),
    "datetime_lst": find_column(
        weather_raw.columns,
        [
            "Date/Time (LST)",
            "Date/Time LST",
            "Date/Time",
        ],
    ),
    "temp_c": find_column(
        weather_raw.columns,
        [
            "Temp (°C)",
            "Temp (C)",
            "Temperature (°C)",
        ],
    ),
    "dew_point_c": find_column(
        weather_raw.columns,
        [
            "Dew Point Temp (°C)",
            "Dew Point Temp (C)",
        ],
    ),
    "rel_hum_pct": find_column(
        weather_raw.columns,
        [
            "Rel Hum (%)",
            "Relative Humidity (%)",
        ],
    ),
    "precip_mm": find_column(
        weather_raw.columns,
        [
            "Precip. Amount (mm)",
            "Precip Amount (mm)",
            "Hourly Precipitation Amount (mm)",
        ],
    ),
    "wind_speed_kmh": find_column(
        weather_raw.columns,
        [
            "Wind Spd (km/h)",
            "Wind Speed (km/h)",
        ],
    ),
    "visibility_km": find_column(
        weather_raw.columns,
        [
            "Visibility (km)",
        ],
    ),
    "station_pressure_kpa": find_column(
        weather_raw.columns,
        [
            "Stn Press (kPa)",
            "Station Pressure (kPa)",
        ],
    ),
    "weather_text": find_column(
        weather_raw.columns,
        [
            "Weather",
        ],
    ),
}

display(
    pd.DataFrame(
        weather_columns.items(),
        columns=["project_field", "source_column"],
    )
)

,project_field,source_column
0,datetime_utc,NaN
1,datetime_lst,Date/Time (LST)
2,temp_c,Temp (°C)
3,dew_point_c,Dew Point Temp (°C)
4,rel_hum_pct,Rel Hum (%)
5,precip_mm,Precip. Amount (mm)
6,wind_speed_kmh,Wind Spd (km/h)
7,visibility_km,Visibility (km)
8,station_pressure_kpa,Stn Press (kPa)
9,weather_text,Weather


In [11]:
# A usable timestamp and temperature field are required.
if (
    weather_columns["datetime_utc"] is None
    and weather_columns["datetime_lst"] is None
):
    raise KeyError(
        "Could not find an ECCC hourly datetime column."
    )

if weather_columns["temp_c"] is None:
    raise KeyError(
        "Could not find the ECCC temperature column."
    )

### Timestamp Handling

ECCC hourly files commonly label observations as **LST** (Local Standard Time).
For Calgary, LST is MST (`UTC−07:00`) year-round.

To avoid daylight-saving mismatches, weather observations are converted to UTC
first and then joined to incidents using the UTC hour.

In [12]:
# Create a clean weather DataFrame.
weather = pd.DataFrame()

if weather_columns["datetime_utc"] is not None:
    weather["weather_hour_utc"] = pd.to_datetime(
        weather_raw[weather_columns["datetime_utc"]],
        errors="coerce",
        utc=True,
    ).dt.floor("h")

else:
    weather_lst = pd.to_datetime(
        weather_raw[weather_columns["datetime_lst"]],
        errors="coerce",
    )

    # Calgary LST is fixed MST, UTC-07:00.
    weather["weather_hour_utc"] = (
        weather_lst
        .dt.tz_localize(
            "Etc/GMT+7",
            ambiguous="NaT",
            nonexistent="NaT",
        )
        .dt.tz_convert("UTC")
        .dt.floor("h")
    )

print(
    "Unparseable weather timestamps:",
    weather["weather_hour_utc"].isna().sum(),
)

Unparseable weather timestamps: 0


In [13]:
# Convert available weather measurements to numeric project fields.
numeric_weather_fields = [
    "temp_c",
    "dew_point_c",
    "rel_hum_pct",
    "precip_mm",
    "wind_speed_kmh",
    "visibility_km",
    "station_pressure_kpa",
]

for field in numeric_weather_fields:
    source_column = weather_columns[field]

    if source_column is None:
        weather[field] = np.nan
    else:
        weather[field] = pd.to_numeric(
            weather_raw[source_column],
            errors="coerce",
        )

# Preserve the source weather description when available.
if weather_columns["weather_text"] is None:
    weather["weather_text"] = ""
else:
    weather["weather_text"] = (
        weather_raw[weather_columns["weather_text"]]
        .astype("string")
        .fillna("")
        .str.strip()
    )

In [14]:
# Remove invalid timestamps and duplicate hourly records.
weather = (
    weather
    .dropna(subset=["weather_hour_utc"])
    .sort_values("weather_hour_utc")
    .drop_duplicates(
        subset="weather_hour_utc",
        keep="first",
    )
    .reset_index(drop=True)
)

# Add Calgary local time for display and analysis.
weather["weather_hour_local"] = (
    weather["weather_hour_utc"]
    .dt.tz_convert("America/Edmonton")
)

print(f"Clean weather hours: {len(weather):,}")
print(
    "Weather period:",
    weather["weather_hour_local"].min(),
    "to",
    weather["weather_hour_local"].max(),
)

Clean weather hours: 86,184
Weather period: 2016-12-01 00:00:00-07:00 to 2026-10-01 00:00:00-06:00


## Weather Feature Engineering

The weather flags below are deliberately simple and interpretable analytical features.

The ECCC hourly precipitation field in this project snapshot is checked for actual availability before it is used. If that field is entirely missing, it is **not** treated as zero precipitation.

In [15]:
# Normalize weather text for keyword matching.
weather_text_lower = (
    weather["weather_text"]
    .astype("string")
    .fillna("")
    .str.lower()
)

# Text-based weather flags. In ECCC hourly data a blank Weather field normally
# means no listed weather phenomenon for that hour.
weather["snow_flag"] = weather_text_lower.str.contains(
    r"snow|blowing snow|ice crystals|freezing rain|freezing drizzle",
    regex=True,
)

weather["rain_flag"] = weather_text_lower.str.contains(
    r"rain|drizzle|shower|thunderstorm",
    regex=True,
)


def measured_boolean(series, predicate):
    """Return a nullable boolean so missing measurements stay unknown."""

    result = pd.Series(
        pd.NA,
        index=series.index,
        dtype="boolean",
    )

    observed = series.notna()
    result.loc[observed] = predicate(
        series.loc[observed]
    ).astype(bool)

    return result


weather["freezing_temp"] = measured_boolean(
    weather["temp_c"],
    lambda s: s <= 0,
)

weather["extreme_cold"] = measured_boolean(
    weather["temp_c"],
    lambda s: s <= -20,
)

weather["low_visibility"] = measured_boolean(
    weather["visibility_km"],
    lambda s: s < 5,
)

weather["high_wind"] = measured_boolean(
    weather["wind_speed_kmh"],
    lambda s: s >= 50,
)

weather["measurable_precip"] = measured_boolean(
    weather["precip_mm"],
    lambda s: s > 0,
)

# Combined project-defined adverse-weather flag.
# True if any observed component is adverse, False if at least one component is
# known and none is adverse, and missing only if every component is unknown.
adverse_components = pd.DataFrame(
    {
        "snow_flag": weather["snow_flag"].astype("boolean"),
        "rain_flag": weather["rain_flag"].astype("boolean"),
        "freezing_temp": weather["freezing_temp"],
        "low_visibility": weather["low_visibility"],
        "high_wind": weather["high_wind"],
        "measurable_precip": weather["measurable_precip"],
    }
)

any_adverse = adverse_components.fillna(False).any(axis=1)
any_known = adverse_components.notna().any(axis=1)

weather["adverse_weather_flag"] = pd.Series(
    pd.NA,
    index=weather.index,
    dtype="boolean",
)

weather.loc[
    any_known & ~any_adverse,
    "adverse_weather_flag",
] = False

weather.loc[
    any_adverse,
    "adverse_weather_flag",
] = True

display(
    weather[
        [
            "weather_hour_local",
            "temp_c",
            "rel_hum_pct",
            "wind_speed_kmh",
            "visibility_km",
            "precip_mm",
            "weather_text",
            "snow_flag",
            "rain_flag",
            "adverse_weather_flag",
        ]
    ].head()
)

,weather_hour_local,temp_c,rel_hum_pct,wind_speed_kmh,visibility_km,precip_mm,weather_text,snow_flag,rain_flag,adverse_weather_flag
0,2016-12-01 00:00:00-07:00,-4.8,91.0,0.0,6.4,NaN,Snow,True,False,True
1,2016-12-01 01:00:00-07:00,-4.8,92.0,0.0,6.4,NaN,Snow,True,False,True
2,2016-12-01 02:00:00-07:00,-4.4,92.0,0.0,3.2,NaN,Snow,True,False,True
3,2016-12-01 03:00:00-07:00,-3.7,93.0,5.0,4.0,NaN,Snow,True,False,True
4,2016-12-01 04:00:00-07:00,-3.5,91.0,5.0,4.8,NaN,Snow,True,False,True


In [16]:
# Review weather missingness.
weather_missing = (
    weather.isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
    .to_frame("missing_percent")
)

display(weather_missing)

,missing_percent
measurable_precip,100.0
precip_mm,100.0
high_wind,0.2
temp_c,0.2
rel_hum_pct,0.2
dew_point_c,0.2
visibility_km,0.2
station_pressure_kpa,0.2
extreme_cold,0.2
wind_speed_kmh,0.2


## Join Weather to Traffic Incidents

Incidents are matched to the corresponding ECCC observation using the UTC hour.

In [17]:
# Round incidents down to their observation hour.
incidents["incident_hour_utc"] = (
    incidents["start_dt_utc"]
    .dt.floor("h")
)

# Keep weather fields with clear names for the merge.
weather_join_columns = [
    "weather_hour_utc",
    "temp_c",
    "dew_point_c",
    "rel_hum_pct",
    "precip_mm",
    "wind_speed_kmh",
    "visibility_km",
    "station_pressure_kpa",
    "weather_text",
    "snow_flag",
    "rain_flag",
    "freezing_temp",
    "extreme_cold",
    "low_visibility",
    "high_wind",
    "measurable_precip",
    "adverse_weather_flag",
]

incidents_weather = incidents.merge(
    weather[weather_join_columns],
    left_on="incident_hour_utc",
    right_on="weather_hour_utc",
    how="left",
    validate="many_to_one",
)

weather_match_rate = (
    incidents_weather["weather_hour_utc"]
    .notna()
    .mean()
    * 100
)

print(
    f"Incident-to-weather match rate: "
    f"{weather_match_rate:.2f}%"
)

Incident-to-weather match rate: 100.00%


In [18]:
# Review weather coverage by incident year.
weather_coverage_by_year = (
    incidents_weather
    .assign(
        weather_matched=(
            incidents_weather["weather_hour_utc"]
            .notna()
        )
    )
    .groupby("year")
    .agg(
        incidents=("incident_id", "size"),
        weather_matched=("weather_matched", "sum"),
    )
    .reset_index()
)

weather_coverage_by_year["match_percent"] = (
    weather_coverage_by_year["weather_matched"]
    / weather_coverage_by_year["incidents"]
    * 100
).round(2)

display(weather_coverage_by_year)

,year,incidents,weather_matched,match_percent
0,2016,311,311,100.0
1,2017,5273,5273,100.0
2,2018,6567,6567,100.0
3,2019,4624,4624,100.0
4,2020,5783,5783,100.0
5,2021,6990,6990,100.0
6,2022,7572,7572,100.0
7,2023,7229,7229,100.0
8,2024,7493,7493,100.0
9,2025,7005,7005,100.0


## Build an Hourly Exposure Dataset

Comparing only incidents that occurred in snow versus rain would ignore how often each weather condition actually occurs.

The hourly weather observations therefore provide the exposure frame. Hours with no recorded incident receive `incident_count = 0`.

The City of Calgary dataset also warns that archive gaps can occur because of system or script malfunction. To avoid treating obvious collection outages as genuine zero-incident periods, this notebook marks any interval longer than **48 hours between consecutive recorded incidents** as an inferred collection gap. Those hours are retained in the exported hourly dataset but excluded from rate comparisons and predictive modelling.

The 48-hour rule is a conservative project heuristic, not an official City-provided uptime flag.

In [19]:
# Count incidents during every UTC hour.
hourly_incident_counts = (
    incidents
    .groupby("incident_hour_utc")
    .size()
    .rename("incident_count")
    .reset_index()
)

# Weather provides the hourly exposure frame.
hourly_incident_weather = weather.merge(
    hourly_incident_counts,
    left_on="weather_hour_utc",
    right_on="incident_hour_utc",
    how="left",
)

hourly_incident_weather["incident_count"] = (
    hourly_incident_weather["incident_count"]
    .fillna(0)
    .astype(int)
)

hourly_incident_weather["year"] = (
    hourly_incident_weather["weather_hour_local"]
    .dt.year
)

hourly_incident_weather["month"] = (
    hourly_incident_weather["weather_hour_local"]
    .dt.month
)

hourly_incident_weather["weekday_number"] = (
    hourly_incident_weather["weather_hour_local"]
    .dt.weekday
)

hourly_incident_weather["weekday"] = (
    hourly_incident_weather["weather_hour_local"]
    .dt.day_name()
)

hourly_incident_weather["hour"] = (
    hourly_incident_weather["weather_hour_local"]
    .dt.hour
)

hourly_incident_weather["weekend"] = (
    hourly_incident_weather["weekday_number"] >= 5
)

# Infer obvious source-collection gaps from unusually long periods
# with no recorded incidents anywhere in Calgary.
COLLECTION_GAP_THRESHOLD_HOURS = 48

incident_times = (
    incidents["start_dt_utc"]
    .dropna()
    .sort_values()
    .drop_duplicates()
    .reset_index(drop=True)
)

gap_table = pd.DataFrame(
    {
        "gap_start_utc": (
            incident_times.iloc[:-1]
            .reset_index(drop=True)
        ),
        "gap_end_utc": (
            incident_times.iloc[1:]
            .reset_index(drop=True)
        ),
    }
)

gap_table["gap_hours"] = (
    (
        gap_table["gap_end_utc"]
        - gap_table["gap_start_utc"]
    )
    .dt.total_seconds()
    / 3600
)

large_collection_gaps = (
    gap_table[
        gap_table["gap_hours"]
        > COLLECTION_GAP_THRESHOLD_HOURS
    ]
    .copy()
    .reset_index(drop=True)
)

hourly_incident_weather[
    "inferred_collection_gap"
] = False

for gap in large_collection_gaps.itertuples(
    index=False
):
    in_gap = (
        (
            hourly_incident_weather[
                "weather_hour_utc"
            ] > gap.gap_start_utc
        )
        & (
            hourly_incident_weather[
                "weather_hour_utc"
            ] < gap.gap_end_utc
        )
    )

    hourly_incident_weather.loc[
        in_gap,
        "inferred_collection_gap",
    ] = True

gap_display = large_collection_gaps.copy()

if not gap_display.empty:
    gap_display["gap_start_local"] = (
        gap_display["gap_start_utc"]
        .dt.tz_convert("America/Edmonton")
    )
    gap_display["gap_end_local"] = (
        gap_display["gap_end_utc"]
        .dt.tz_convert("America/Edmonton")
    )

    display(
        gap_display[
            [
                "gap_start_local",
                "gap_end_local",
                "gap_hours",
            ]
        ]
    )

print(
    f"Hourly exposure rows: "
    f"{len(hourly_incident_weather):,}"
)

print(
    "Hours marked as inferred collection gaps:",
    f"{hourly_incident_weather['inferred_collection_gap'].sum():,}",
)

,gap_start_local,gap_end_local,gap_hours
0,2016-12-20 10:51:00-07:00,2017-02-08 08:12:00-07:00,1197.350000
1,2019-04-18 15:35:00-06:00,2019-04-29 13:10:00-06:00,261.583333
2,2019-06-13 21:32:00-06:00,2019-08-06 06:51:00-06:00,1281.316667
3,2019-08-06 06:51:00-06:00,2019-09-16 10:03:00-06:00,987.200000
4,2020-10-16 15:21:00-06:00,2020-10-20 08:15:00-06:00,88.900000
5,2025-01-19 03:03:00-07:00,2025-01-21 07:10:00-07:00,52.116667
6,2025-05-20 12:08:00-06:00,2025-05-23 12:33:00-06:00,72.416667
7,2025-05-23 12:33:00-06:00,2025-05-28 09:10:00-06:00,116.616667
8,2025-05-28 11:59:00-06:00,2025-07-02 09:03:00-06:00,837.066667


Hourly exposure rows: 86,184
Hours marked as inferred collection gaps: 4,897


### Main Analysis Window

The main weather-rate analysis uses calendar years **2017–2025**, but it no longer assumes that every hour in those years was observed correctly.

Hours inside inferred source-collection gaps are excluded. This prevents known archive outages from being counted as genuine zero-incident exposure hours.

In [20]:
# Use the main calendar-year window while excluding inferred collection gaps.
analysis_hourly = hourly_incident_weather[
    (
        hourly_incident_weather["year"]
        .between(2017, 2025)
    )
    & (
        ~hourly_incident_weather[
            "inferred_collection_gap"
        ]
    )
].copy()

print(
    "Analysis period:",
    analysis_hourly["weather_hour_local"].min(),
    "to",
    analysis_hourly["weather_hour_local"].max(),
)

print(
    "Eligible analysis hours:",
    f"{len(analysis_hourly):,}",
)

print(
    "Excluded inferred-gap hours:",
    f"{(
        hourly_incident_weather[
            hourly_incident_weather['year']
            .between(2017, 2025)
        ]['inferred_collection_gap']
        .sum()
    ):,}",
)

Analysis period: 2017-02-08 09:00:00-07:00 to 2025-12-31 23:00:00-07:00
Eligible analysis hours: 74,268
Excluded inferred-gap hours: 4,620


In [21]:
def condition_rate_table(df, column):
    """Summarize incident frequency using only hours where a condition is known."""

    known = df[
        df[column].notna()
    ].copy()

    result = (
        known
        .groupby(column)
        .agg(
            observed_hours=("incident_count", "size"),
            total_incidents=("incident_count", "sum"),
            mean_incidents_per_hour=("incident_count", "mean"),
        )
        .reset_index()
    )

    result["mean_incidents_per_hour"] = (
        result["mean_incidents_per_hour"]
        .round(4)
    )

    return result

## Weather and Incident Frequency

In [22]:
# Compare several interpretable weather conditions.
condition_columns = [
    "snow_flag",
    "rain_flag",
    "freezing_temp",
    "low_visibility",
    "high_wind",
    "adverse_weather_flag",
]

# Only analyze measured precipitation if the source field
# contains at least one real observation.
if analysis_hourly["precip_mm"].notna().any():
    condition_columns.insert(
        -1,
        "measurable_precip",
    )

weather_condition_tables = {}

for column in condition_columns:
    weather_condition_tables[column] = (
        condition_rate_table(
            analysis_hourly,
            column,
        )
    )

    print("\n", column)
    display(weather_condition_tables[column])


 snow_flag


,snow_flag,observed_hours,total_incidents,mean_incidents_per_hour
0,False,67685,51059,0.7544
1,True,6583,7464,1.1338



 rain_flag


,rain_flag,observed_hours,total_incidents,mean_incidents_per_hour
0,False,70792,55771,0.7878
1,True,3476,2752,0.7917



 freezing_temp


...

,low_visibility,observed_hours,total_incidents,mean_incidents_per_hour
0,False,68952,52672,0.7639
1,True,5241,5825,1.1114



 high_wind


,high_wind,observed_hours,total_incidents,mean_incidents_per_hour
0,False,74066,58345,0.7877
1,True,124,147,1.1855



 adverse_weather_flag


,adverse_weather_flag,observed_hours,total_incidents,mean_incidents_per_hour
0,False,46965,35618,0.7584
1,True,27303,22905,0.8389


In [23]:
# Create a compact comparison table for dashboard use.
condition_labels = {
    "snow_flag": "Snow / freezing-precipitation text",
    "rain_flag": "Rain / drizzle / thunderstorm text",
    "freezing_temp": "Temperature ≤ 0°C",
    "low_visibility": "Visibility < 5 km",
    "high_wind": "Wind speed ≥ 50 km/h",
    "measurable_precip": "Measured precipitation > 0 mm",
    "adverse_weather_flag": "Any project adverse-weather condition",
}

condition_comparison_rows = []

for column, table in weather_condition_tables.items():
    label = condition_labels[column]

    true_row = table[
        table[column] == True
    ]

    false_row = table[
        table[column] == False
    ]

    if true_row.empty or false_row.empty:
        continue

    true_rate = float(
        true_row[
            "mean_incidents_per_hour"
        ].iloc[0]
    )

    false_rate = float(
        false_row[
            "mean_incidents_per_hour"
        ].iloc[0]
    )

    relative_rate = (
        true_rate / false_rate
        if false_rate > 0
        else np.nan
    )

    condition_comparison_rows.append(
        {
            "condition": label,
            "condition_hours": int(
                true_row[
                    "observed_hours"
                ].iloc[0]
            ),
            "mean_incidents_per_hour_condition":
                true_rate,
            "mean_incidents_per_hour_other":
                false_rate,
            "relative_rate": relative_rate,
        }
    )

condition_comparison = pd.DataFrame(
    condition_comparison_rows
)

display(condition_comparison)

,condition,condition_hours,mean_incidents_per_hour_condition,mean_incidents_per_hour_other,relative_rate
0,Snow / freezing-precipitation text,6583,1.1338,0.7544,1.502916
1,Rain / drizzle / thunderstorm text,3476,0.7917,0.7878,1.004950
2,Temperature ≤ 0°C,23459,0.8593,0.7557,1.137091
3,Visibility < 5 km,5241,1.1114,0.7639,1.454902
4,Wind speed ≥ 50 km/h,124,1.1855,0.7877,1.505015
5,Any project adverse-weather condition,27303,0.8389,0.7584,1.106145


In [24]:
# Plot relative incident-frequency differences.
plot_conditions = condition_comparison.dropna(
    subset=["relative_rate"]
).copy()

fig = px.bar(
    plot_conditions,
    x="condition",
    y="relative_rate",
    title=(
        "Relative Recorded Incident Frequency "
        "by Weather Condition"
    ),
    labels={
        "condition": "Weather condition",
        "relative_rate": (
            "Mean incidents/hour relative to other hours"
        ),
    },
)

fig.add_hline(
    y=1,
    line_dash="dash",
)

fig.show()

A relative rate above `1.0` means the observed incident count per hour was higher
during that condition than during hours without it. This is descriptive
association, not proof that the weather condition caused the incidents.

## Temperature Bands

In [25]:
# Group temperature into interpretable ranges.
temperature_bins = [
    -np.inf,
    -20,
    -10,
    0,
    10,
    20,
    30,
    np.inf,
]

temperature_labels = [
    "< -20°C",
    "-20 to -10°C",
    "-10 to 0°C",
    "0 to 10°C",
    "10 to 20°C",
    "20 to 30°C",
    "≥ 30°C",
]

analysis_hourly["temperature_band"] = pd.cut(
    analysis_hourly["temp_c"],
    bins=temperature_bins,
    labels=temperature_labels,
    right=False,
)

temperature_rates = (
    analysis_hourly
    .dropna(subset=["temperature_band"])
    .groupby(
        "temperature_band",
        observed=True,
    )
    .agg(
        observed_hours=("incident_count", "size"),
        total_incidents=("incident_count", "sum"),
        mean_incidents_per_hour=("incident_count", "mean"),
    )
    .reset_index()
)

display(temperature_rates)

,temperature_band,observed_hours,total_incidents,mean_incidents_per_hour
0,< -20°C,2232,2934,1.314516
1,-20 to -10°C,6023,5837,0.969118
2,-10 to 0°C,14922,11234,0.752848
3,0 to 10°C,23646,15606,0.659985
4,10 to 20°C,20208,14823,0.733521
5,20 to 30°C,6801,7537,1.108219
6,≥ 30°C,359,524,1.459610


In [26]:
# Plot incident frequency by temperature band.
fig = px.bar(
    temperature_rates,
    x="temperature_band",
    y="mean_incidents_per_hour",
    title="Mean Recorded Incidents per Hour by Temperature Band",
    labels={
        "temperature_band": "Temperature",
        "mean_incidents_per_hour": "Mean recorded incidents per hour",
    },
)

fig.show()

## Time-Specific Hotspot Profiles

This table allows the final dashboard to answer questions such as:

- Which hotspots are most active during morning rush?
- Which hotspots are more common on Friday evenings?
- How does hotspot activity change by hour or weekday?

In [27]:
# Use only incidents that belong to a DBSCAN hotspot.
clustered_weather_incidents = incidents_weather[
    incidents_weather["hotspot_id"] != -1
].copy()

hotspot_time_profile = (
    clustered_weather_incidents
    .groupby(
        [
            "hotspot_id",
            "weekday",
            "weekday_number",
            "hour",
            "time_period",
        ],
        dropna=False,
    )
    .agg(
        incident_count=("incident_id", "size"),
        adverse_weather_observed=(
            "adverse_weather_flag",
            "count",
        ),
        adverse_weather_incidents=(
            "adverse_weather_flag",
            "sum",
        ),
    )
    .reset_index()
)

hotspot_time_profile["adverse_weather_share"] = np.where(
    hotspot_time_profile["adverse_weather_observed"] > 0,
    hotspot_time_profile["adverse_weather_incidents"]
    / hotspot_time_profile["adverse_weather_observed"],
    np.nan,
)

display(
    hotspot_time_profile
    .sort_values(
        "incident_count",
        ascending=False,
    )
    .head(20)
)

,hotspot_id,weekday,weekday_number,hour,time_period,incident_count,adverse_weather_observed,adverse_weather_incidents,adverse_weather_share
14,0,Friday,4,17,Evening Rush,32,32,10,0.312500
105,0,Thursday,3,17,Evening Rush,29,29,9,0.310345
150,0,Wednesday,2,16,Evening Rush,28,28,8,0.285714
977,13,Wednesday,2,17,Evening Rush,26,26,4,0.153846
939,13,Thursday,3,15,Evening Rush,23,23,6,0.260870
865,13,Friday,4,14,Midday,22,22,7,0.318182
151,0,Wednesday,2,17,Evening Rush,22,22,8,0.363636
13,0,Friday,4,16,Evening Rush,22,22,6,0.272727
2053,33,Saturday,5,13,Midday,21,21,7,0.333333
961,13,Tuesday,1,17,Evening Rush,19,19,3,0.157895


## Historical Incident Risk Index

The project creates one transparent score for each traffic-control location.

The index is **not an official safety rating** and does not estimate collision probability. It summarizes historical recorded-incident patterns into a 0–100 analytical score for exploration.

### Components

| Component | Weight |
|---|---:|
| Historical matched incident count | 30% |
| Recent matched incident count (2024–2025) | 25% |
| Share of incidents inside DBSCAN hotspots | 15% |
| Adverse-weather incident share | 10% |
| Share during morning/evening rush periods | 10% |
| Positive coverage-adjusted historical trend | 10% |

Count features are based on recorded incidents. The trend component uses incidents per observed coverage hour so large archive gaps do not directly create artificial downward years.

In [28]:
# Keep incidents that were accepted by the 75 m matching threshold.
matched_weather = incidents_weather[
    incidents_weather["matched_to_signal"] == True
].copy()

# Use a consistent interior calendar-year window for the risk index so the
# partial 2016 and 2026 boundary years do not affect its components.
RISK_START_YEAR = 2017
RISK_END_YEAR = 2025

matched_risk = matched_weather[
    matched_weather["year"].between(
        RISK_START_YEAR,
        RISK_END_YEAR,
    )
].copy()

print(
    "Matched incidents used for risk features:",
    f"{len(matched_risk):,}",
)
print(
    "Risk-index calendar window:",
    f"{RISK_START_YEAR}–{RISK_END_YEAR}",
)

Matched incidents used for risk features: 30,163
Risk-index calendar window: 2017–2025


In [29]:
# Base historical traffic-control features within the risk-index window.
control_features = (
    matched_risk
    .groupby("matched_signal_id")
    .agg(
        historical_incident_count=("incident_id", "size"),
        hotspot_incident_share=(
            "hotspot_id",
            lambda s: (s != -1).mean(),
        ),
        rush_hour_share=(
            "time_period",
            lambda s: s.isin(
                ["Morning Rush", "Evening Rush"]
            ).mean(),
        ),
        winter_share=(
            "season",
            lambda s: (s == "Winter").mean(),
        ),
        adverse_weather_share=(
            "adverse_weather_flag",
            lambda s: s.dropna().mean()
            if s.notna().any()
            else np.nan,
        ),
    )
    .reset_index()
    .rename(
        columns={
            "matched_signal_id": "signal_id",
        }
    )
)

control_features["signal_id"] = (
    control_features["signal_id"]
    .astype("Int64")
)

display(control_features.head())

,signal_id,historical_incident_count,hotspot_incident_share,rush_hour_share,winter_share,adverse_weather_share
0,1109000525,67,0.880597,0.522388,0.179104,0.328358
1,1109000526,10,0.000000,0.500000,0.200000,0.4
2,1109000527,3,0.000000,0.333333,0.333333,0.0
3,1109000529,4,0.000000,0.250000,0.000000,0.25
4,1109000539,2,0.000000,1.000000,0.500000,0.0


In [30]:
# Recent activity uses the two latest years in the risk-index window.
# These are recorded archive counts and still inherit the source's limitations.
recent_incidents = matched_risk[
    matched_risk["year"].between(
        2024,
        2025,
    )
]

recent_counts = (
    recent_incidents
    .groupby("matched_signal_id")
    .size()
    .rename("recent_incident_count")
    .reset_index()
    .rename(
        columns={
            "matched_signal_id": "signal_id",
        }
    )
)

recent_counts["signal_id"] = (
    recent_counts["signal_id"]
    .astype("Int64")
)

control_features = control_features.merge(
    recent_counts,
    on="signal_id",
    how="left",
)

control_features["recent_incident_count"] = (
    control_features["recent_incident_count"]
    .fillna(0)
    .astype(int)
)

In [31]:
# Calculate a coverage-adjusted annual incident trend for each matched traffic control.
trend_years = np.arange(
    RISK_START_YEAR,
    RISK_END_YEAR + 1,
)

annual_coverage_hours = (
    analysis_hourly
    .groupby("year")
    .size()
    .reindex(trend_years)
)

if annual_coverage_hours.isna().any() or (annual_coverage_hours <= 0).any():
    raise ValueError(
        "Every risk-index year must have eligible observed weather hours."
    )

# Match the numerator to the same eligible hourly exposure frame used in the denominator.
eligible_exposure_hours = set(
    analysis_hourly["weather_hour_utc"]
)

trend_incidents = matched_risk[
    matched_risk["incident_hour_utc"].isin(
        eligible_exposure_hours
    )
].copy()

annual_control_counts = (
    trend_incidents
    .groupby(
        ["matched_signal_id", "year"]
    )
    .size()
    .rename("count")
    .reset_index()
)


def control_trend_slope(signal_id):
    subset = annual_control_counts[
        annual_control_counts[
            "matched_signal_id"
        ] == signal_id
    ]

    counts_by_year = (
        subset
        .set_index("year")["count"]
        .reindex(
            trend_years,
            fill_value=0,
        )
        .astype(float)
    )

    rates_per_1000_hours = (
        counts_by_year
        / annual_coverage_hours
        * 1000
    )

    values = rates_per_1000_hours.to_numpy(
        dtype=float
    )

    if np.allclose(values, values[0]):
        return 0.0

    return float(
        np.polyfit(
            trend_years,
            values,
            deg=1,
        )[0]
    )


control_features["annual_trend_slope"] = (
    control_features["signal_id"]
    .apply(control_trend_slope)
)

print("Annual eligible coverage hours:")
display(
    annual_coverage_hours
    .rename("eligible_hours")
    .to_frame()
)

Annual eligible coverage hours:


,eligible_hours
year,
2017,7839
2018,8760
2019,6229
2020,8695
2021,8760
2022,8760
2023,8760
2024,8784
2025,7681


In [32]:
# Merge features onto every traffic-control location.
risk_controls = (
    traffic_controls
    .merge(
        control_features,
        on="signal_id",
        how="left",
    )
)

fill_zero_columns = [
    "historical_incident_count",
    "recent_incident_count",
    "hotspot_incident_share",
    "rush_hour_share",
    "winter_share",
    "annual_trend_slope",
]

for column in fill_zero_columns:
    risk_controls[column] = (
        risk_controls[column]
        .fillna(0)
    )

# No matched weather means no observed adverse-weather share.
risk_controls["adverse_weather_share"] = (
    risk_controls["adverse_weather_share"]
    .fillna(0)
)

In [33]:
def scale_0_100(series, log_transform=False):
    """Scale a non-negative feature to 0–100 with 99th-percentile clipping."""

    values = (
        pd.to_numeric(
            series,
            errors="coerce",
        )
        .fillna(0)
        .clip(lower=0)
        .astype(float)
    )

    if log_transform:
        values = np.log1p(values)

    upper = values.quantile(0.99)

    if upper <= 0:
        return pd.Series(
            0.0,
            index=series.index,
        )

    clipped = values.clip(
        upper=upper,
    )

    return (
        clipped / upper * 100
    ).clip(0, 100)

In [34]:
# Create normalized component scores.
risk_controls["historical_score"] = scale_0_100(
    risk_controls["historical_incident_count"],
    log_transform=True,
)

risk_controls["recent_score"] = scale_0_100(
    risk_controls["recent_incident_count"],
    log_transform=True,
)

risk_controls["hotspot_score"] = (
    risk_controls["hotspot_incident_share"]
    .clip(0, 1)
    * 100
)

risk_controls["adverse_weather_score"] = (
    risk_controls["adverse_weather_share"]
    .clip(0, 1)
    * 100
)

risk_controls["rush_hour_score"] = (
    risk_controls["rush_hour_share"]
    .clip(0, 1)
    * 100
)

risk_controls["trend_score"] = scale_0_100(
    risk_controls["annual_trend_slope"]
    .clip(lower=0),
)

In [35]:
# Weighted Historical Incident Risk Index.
risk_controls["historical_incident_risk_index"] = (
    0.30 * risk_controls["historical_score"]
    + 0.25 * risk_controls["recent_score"]
    + 0.15 * risk_controls["hotspot_score"]
    + 0.10 * risk_controls["adverse_weather_score"]
    + 0.10 * risk_controls["rush_hour_score"]
    + 0.10 * risk_controls["trend_score"]
).round(1)

display(
    risk_controls[
        [
            "signal_id",
            "intersection_name",
            "control_type",
            "historical_incident_count",
            "recent_incident_count",
            "hotspot_incident_share",
            "adverse_weather_share",
            "rush_hour_share",
            "annual_trend_slope",
            "historical_incident_risk_index",
        ]
    ]
    .sort_values(
        "historical_incident_risk_index",
        ascending=False,
    )
    .head(20)
)

,signal_id,intersection_name,control_type,historical_incident_count,recent_incident_count,hotspot_incident_share,adverse_weather_share,rush_hour_share,annual_trend_slope,historical_incident_risk_index
30,1109001768,61 AVENUE & 52 STREET,Traffic signal,87.0,30.0,1.000000,0.482759,0.620690,0.135788,90.0
10,1110004236,88 AVENUE & METIS TRAIL,Traffic signal,115.0,42.0,1.000000,0.4,0.530435,0.252421,89.3
4,1109001451,MEMORIAL DRIVE & DEERFOOT TRAIL SB RAMP,Traffic signal,168.0,64.0,1.000000,0.35119,0.541667,0.393212,88.9
21,1109001415,McKNIGHT BOULEVARD & AVIATION BLVD/12 STREET,Traffic signal,100.0,35.0,1.000000,0.33,0.520000,0.192300,88.4
2,1109001467,16 AVENUE & 19 STREET,Traffic signal,201.0,46.0,1.000000,0.353234,0.492537,0.100779,87.9
19,1110166328,88 AVENUE & 60 STREET,Traffic signal,99.0,33.0,1.000000,0.323232,0.444444,0.287967,87.5
18,1109001776,RUNDLEHORN DRIVE & 52 STREET,Traffic signal,104.0,30.0,1.000000,0.365385,0.461538,0.096139,87.3
15,1109750648,104 AVENUE & METIS TRAIL,Traffic signal,104.0,30.0,1.000000,0.230769,0.490385,0.115241,87.2
57,1109001438,16 AVENUE WB & DEERFOOT TRAIL NB RAMP,Traffic signal,73.0,22.0,1.000000,0.410959,0.506849,0.105832,86.1
50,1109001243,COUNTRY HILLS BOULEVARD & HARVEST HILLS BOULEVARD,Traffic signal,71.0,23.0,1.000000,0.394366,0.549296,0.097264,85.7


In [36]:
# Create a simple explanation from the largest weighted components.
component_info = {
    "historical": (
        "Long-term incident history",
        "historical_score",
        0.30,
    ),
    "recent": (
        "Recent incident activity",
        "recent_score",
        0.25,
    ),
    "hotspot": (
        "DBSCAN hotspot concentration",
        "hotspot_score",
        0.15,
    ),
    "weather": (
        "Adverse-weather association",
        "adverse_weather_score",
        0.10,
    ),
    "rush": (
        "Rush-hour concentration",
        "rush_hour_score",
        0.10,
    ),
    "trend": (
        "Increasing coverage-adjusted trend",
        "trend_score",
        0.10,
    ),
}

def explain_risk_row(row):
    contributions = []

    for _, (label, score_column, weight) in component_info.items():
        contribution = row[score_column] * weight

        contributions.append(
            (label, contribution)
        )

    contributions.sort(
        key=lambda item: item[1],
        reverse=True,
    )

    return "; ".join(
        label
        for label, contribution
        in contributions[:3]
        if contribution > 0
    )

risk_controls["top_risk_factors"] = (
    risk_controls.apply(
        explain_risk_row,
        axis=1,
    )
)

display(
    risk_controls[
        [
            "intersection_name",
            "historical_incident_risk_index",
            "top_risk_factors",
        ]
    ]
    .sort_values(
        "historical_incident_risk_index",
        ascending=False,
    )
    .head(15)
)

,intersection_name,historical_incident_risk_index,top_risk_factors
30,61 AVENUE & 52 STREET,90.0,Long-term incident history; Recent incident ac...
10,88 AVENUE & METIS TRAIL,89.3,Long-term incident history; Recent incident ac...
4,MEMORIAL DRIVE & DEERFOOT TRAIL SB RAMP,88.9,Long-term incident history; Recent incident ac...
21,McKNIGHT BOULEVARD & AVIATION BLVD/12 STREET,88.4,Long-term incident history; Recent incident ac...
2,16 AVENUE & 19 STREET,87.9,Long-term incident history; Recent incident ac...
19,88 AVENUE & 60 STREET,87.5,Long-term incident history; Recent incident ac...
18,RUNDLEHORN DRIVE & 52 STREET,87.3,Long-term incident history; Recent incident ac...
15,104 AVENUE & METIS TRAIL,87.2,Long-term incident history; Recent incident ac...
57,16 AVENUE WB & DEERFOOT TRAIL NB RAMP,86.1,Long-term incident history; Recent incident ac...
50,COUNTRY HILLS BOULEVARD & HARVEST HILLS BOULEVARD,85.7,Long-term incident history; Recent incident ac...


In [37]:
# Plot the highest index values for exploration.
top_risk_controls = (
    risk_controls
    .sort_values(
        "historical_incident_risk_index",
        ascending=False,
    )
    .head(20)
    .copy()
)

fig = px.bar(
    top_risk_controls.sort_values(
        "historical_incident_risk_index"
    ),
    x="historical_incident_risk_index",
    y="intersection_name",
    orientation="h",
    title=(
        "Highest Historical Incident Risk Index Values "
        "(Project Metric)"
    ),
    labels={
        "historical_incident_risk_index":
            "Historical Incident Risk Index",
        "intersection_name":
            "Traffic-control location",
    },
)

fig.show()

The index is intentionally transparent and deterministic. A higher score means
the location has a stronger combination of the project's historical incident
features. It does **not** mean the location has a known probability of collision,
and it does not account for traffic volume or the number of vehicles exposed.

## Validation

In [38]:
# Validate the main derived datasets.
assert weather["weather_hour_utc"].is_unique
assert len(incidents_weather) == len(incidents)

assert hourly_incident_weather[
    "weather_hour_utc"
].is_unique

assert (
    hourly_incident_weather[
        "incident_count"
    ] >= 0
).all()

assert not analysis_hourly[
    "inferred_collection_gap"
].any()

assert risk_controls["signal_id"].is_unique

assert risk_controls[
    "historical_incident_risk_index"
].between(0, 100).all()

print("Weather and risk validation passed.")

Weather and risk validation passed.


## Export Weather and Risk Outputs

In [39]:
# Stable local outputs for later modelling plus compact outputs for the dashboard.
WEATHER_CLEAN_FILE = (
    CLEAN_DIR / "weather_hourly_calgary.csv"
)

INCIDENTS_WEATHER_FILE = (
    CLEAN_DIR / "incidents_weather.csv"
)

HOURLY_WEATHER_FILE = (
    CLEAN_DIR / "hourly_incident_weather.csv"
)

WEATHER_COMPARISON_FILE = (
    CLEAN_DIR / "weather_condition_comparison.csv"
)

TEMPERATURE_BAND_FILE = (
    CLEAN_DIR / "temperature_band_summary.csv"
)

HOTSPOT_TIME_FILE = (
    CLEAN_DIR / "hotspot_time_profile.csv"
)

RISK_CONTROL_FILE = (
    CLEAN_DIR / "traffic_control_risk_summary.csv"
)

DASHBOARD_INCIDENTS_FILE = (
    CLEAN_DIR / "dashboard_incidents.csv"
)

# Compact incident data used by Streamlit. The dashboard does not need the
# full incident-weather join, nearest-control columns, or raw coordinates.
dashboard_incidents = incidents[
    [
        "incident_id",
        "description",
        "start_dt_local",
        "year",
        "month",
        "month_name",
        "weekday",
        "hour",
        "time_period",
        "hotspot_id",
    ]
].copy()

weather.to_csv(
    WEATHER_CLEAN_FILE,
    index=False,
)

incidents_weather.to_csv(
    INCIDENTS_WEATHER_FILE,
    index=False,
)

hourly_incident_weather.to_csv(
    HOURLY_WEATHER_FILE,
    index=False,
)

condition_comparison.to_csv(
    WEATHER_COMPARISON_FILE,
    index=False,
)

temperature_rates.to_csv(
    TEMPERATURE_BAND_FILE,
    index=False,
)

hotspot_time_profile.to_csv(
    HOTSPOT_TIME_FILE,
    index=False,
)

risk_controls.to_csv(
    RISK_CONTROL_FILE,
    index=False,
)

dashboard_incidents.to_csv(
    DASHBOARD_INCIDENTS_FILE,
    index=False,
)

print("Saved local analysis outputs:")
print(WEATHER_CLEAN_FILE)
print(INCIDENTS_WEATHER_FILE)
print(HOURLY_WEATHER_FILE)

print("\nSaved compact dashboard outputs:")
print(DASHBOARD_INCIDENTS_FILE)
print(WEATHER_COMPARISON_FILE)
print(TEMPERATURE_BAND_FILE)
print(HOTSPOT_TIME_FILE)
print(RISK_CONTROL_FILE)

Saved local analysis outputs:
c:\Users\ahnaf\Desktop\calgary-traffic-risk-intelligence\data\cleaned\weather_hourly_calgary.csv
c:\Users\ahnaf\Desktop\calgary-traffic-risk-intelligence\data\cleaned\incidents_weather.csv
c:\Users\ahnaf\Desktop\calgary-traffic-risk-intelligence\data\cleaned\hourly_incident_weather.csv

Saved compact dashboard outputs:
c:\Users\ahnaf\Desktop\calgary-traffic-risk-intelligence\data\cleaned\dashboard_incidents.csv
c:\Users\ahnaf\Desktop\calgary-traffic-risk-intelligence\data\cleaned\weather_condition_comparison.csv
c:\Users\ahnaf\Desktop\calgary-traffic-risk-intelligence\data\cleaned\temperature_band_summary.csv
c:\Users\ahnaf\Desktop\calgary-traffic-risk-intelligence\data\cleaned\hotspot_time_profile.csv
c:\Users\ahnaf\Desktop\calgary-traffic-risk-intelligence\data\cleaned\traffic_control_risk_summary.csv


## Weather and Risk Analysis Summary

This stage adds weather exposure and interpretable traffic-control features to the project.

### Completed

- automated monthly ECCC hourly-weather retrieval
- local caching of raw government weather files
- robust weather-column detection and cleaning
- LST-to-UTC timestamp alignment
- hourly weather feature engineering
- incident-to-weather joining
- complete hourly exposure frame with zero-incident hours
- explicit detection of obvious source-collection gaps
- weather-condition incident-frequency comparisons using eligible exposure hours
- temperature-band analysis
- time-specific hotspot profiles
- traffic-control-level feature engineering
- transparent 0–100 Historical Incident Risk Index
- coverage-adjusted historical trend component
- export of modelling/dashboard-ready datasets

### Important Limitations

- The City traffic-incidents source is an unofficial archive and can contain collection gaps.
- The 48-hour collection-gap rule is a project heuristic, not an official uptime indicator.
- Airport weather is a citywide proxy, not exact conditions at every incident.
- Weather associations do not establish causation.
- The risk index is a project metric, not an official City of Calgary rating.
- Traffic volume and vehicle exposure are not available.
- In this project snapshot, the ECCC hourly precipitation field is not sufficiently populated for predictive use.

The next notebook builds a chronological predictive model using only hours that are not inside inferred collection gaps.